# Studi Kasus

Platform marketplace kita mendeteksi kejanggalan pada laporan penjualan bulanan: total transaksi yang
dilaporkan tim IT (515) tidak sama dengan total yang dipakai tim Finance (490). Sebagai calon data
engineer, jelaskan kepada tim Finance:
1. Mengapa kedua angka tersebut bisa berbeda, dikaitkan dengan proses yang baru saja Anda
lakukan.
> Angka 515 dari laporan tim IT merepresentasikan raw data yang baru ditarik langsung dari sistem pada tahap akuisisi. Sementara itu, angka 490 yang digunakan oleh tim Finance adalah dataset yang telah dibersihkan melalui tahap pra-pemrosesan. Selisih 25 baris tersebut hilang karena 15 baris terdeteksi sebagai transaksi ganda (duplikat) dan 10 baris dihapus (dropna) karena tidak memiliki informasi wajib, seperti nama pelanggan dan metode pembayaran.
2. Apakah 490 baris “lebih benar” dibanding 515 baris? Jelaskan dengan mengaitkan ke konsep Veracity.
> Dataset 490 baris "lebih benar" karena memiliki tingkat Veracity yang jauh lebih baik. Dimensi Veracity dalam Big Data menilai seberapa akurat dan dapat dipercayanya suatu data untuk dianalisis. Jika tim Finance memaksakan penggunaan 515 baris data mentah, akan terjadi prinsip garbage in, garbage out. Transaksi duplikat akan membuat perhitungan pendapatan membengkak dari yang seharusnya, dan baris tanpa informasi pembayaran tidak akan bisa direkonsiliasi.
3. Bagaimana Anda akan menjelaskan keputusan membiarkan kolom rating tetap memiliki missing
value kepada tim Finance yang ingin tahu “rating rata-rata semua transaksi”?
> Tidak seperti metode pembayaran yang wajib ada, pemberian rating oleh pembeli bersifat opsional (boleh dikosongkan). Jika kita mengisi kolom rating yang kosong dengan angka tebakan atau nilai rata-rata keseluruhan, hal ini akan mendistorsi integritas data dan mengacaukan analisis di tahap selanjutnya. Penjelasan terbaik kepada tim Finance adalah metrik "rating rata-rata semua transaksi" harus dihitung hanya dari populasi transaksi yang memang dinilai oleh pelanggan secara riil, sementara sisanya dibiarkan kosong agar umpan balik pelanggan tetap autentik.

In [2]:
# menghubungkan google drive & menyiapkan struktur folder

from google.colab import drive
drive.mount("/content/drive")

import os
DIR_KERJA = "/content/data"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum2"

os.makedirs(DIR_KERJA, exist_ok=True)
os.makedirs(DIR_SIMPAN, exist_ok=True)
print(os.listdir(DIR_SIMPAN))

Mounted at /content/drive
['transaksi_mentah_praktikum.csv', 'transaksi_bersih_praktikum.csv']


In [5]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 24.7 MB/s eta 0:00:00


In [6]:
# setup latihan
import numpy as np
import pandas as pd
from faker import Faker
import random

In [14]:
# latihan soal 1
SEED = 7
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
  trx_id = f"TRX{i:05d}"
  nama_pelanggan = fake.name()
  produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", " "])
  kategori = random.choice(kategori_produk)
  harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
  qty = random.randint(1, 5)

  #variasi format harga : angka polos, ada "Rp", ada desimal ".0", ada spasi
  harga_variants = [
      str(harga_dasar),
      f"Rp{harga_dasar:,}".replace(",", "."),
      f"{harga_dasar}.0",
      f"{harga_dasar}",
  ]
  harga = random.choice(harga_variants)

  # variasi format tanggal : ISO, DD/MM/YYYY, DD-MM-YYYY
  tgl = fake.date_between(start_date="-90d", end_date="today")
  tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
  tanggal = random.choice(tgl_variants)

  metode = random.choice(metode_bayar)
  if random.random() < 0.3:
    metode = metode.lower()
  if random.random() < 0.2:
    kategori = kategori.upper() + " "

  kota = fake.city()
  rating = random.choice([1, 2, 3, 4, 5, None, None]) #rating opsional

  rows.append({
      "transaction_id": trx_id,
      "customer_name": nama_pelanggan,
      "product_name": produk.strip(),
      "category": kategori,
      "price": harga,
      "quatinty": qty,
      "payment_method": metode,
      "transaction_date": tanggal,
      "shipping_city": kota,
      "rating": rating,
  })

df = pd.DataFrame(rows)

# suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
  idx = df.sample(frac=frac, random_state=SEED).index
  df.loc[idx, col] = np.nan

# duplikasi 15 baris baris (mensimulasikan transaksi yang tercatat dua kali)
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

mentah_lat = os.path.join(DIR_SIMPAN, "transaksi_mentah_latihan.csv")
df.to_csv(mentah_lat, index=False)
print("Sukses menyimpan data mentah ke drive")
print("Jumlah baris:", len(df))
print("=========================")

print(df.isnull().sum())
print("=========================")

# strategi penanganan

df = df.dropna(subset=["customer_name",  "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("tidak diketahui")
print("Baris:", len(df))
print("=========================")

#k4
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))
print("=========================")
#k5

#standarisasi teks kategorikal
for col in ["category", "payment_method", "shipping_city"]:
  df[col] = df[col].astype("string").str.strip().str.title()

# "cod" adalah singkatan, kembalikan ke huruf kapital penuh setelah title case
df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

# korreksi tipe data pada kolom price
def bersihkan_harga(x):
  if pd.isna(x):
    return np.nan
  x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
  try:
    return float(x)
  except ValueError:
    return np.nan

df["price"]  = df["price"].apply(bersihkan_harga)

# standarisasi format tanggal ke YYYY-MM-DD
def parse_tanggal(x):
  for fmt in ["%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"]:
    try:
      return pd.to_datetime(x, format=fmt)
    except ValueError:
      continue
  return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

#finalisasi tipe data
df["quatinty"] = df["quatinty"].astype(int)
df["price"] = df["price"].astype(float)

#k6
bersih_lat = os.path.join(DIR_SIMPAN, "transaksi_bersih_latihan.csv")
df.to_csv(bersih_lat, index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Sukses menyimpan data mentah ke drive
Jumlah baris: 515
transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quatinty              0
payment_method       16
transaction_date      0
shipping_city        30
rating              120
dtype: int64
Baris: 495
Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490
Dataset bersih tersimpan: 490 baris


In [10]:
# latihan 2
# Menambahkan kolom is_valid_price (True jika > 0, False jika sebaliknya)
df['is_valid_price'] = df['price'] > 0

# Mengecek jumlah dan menampilkan data harga yang tidak valid
invalid_prices = df[~df['is_valid_price']]
print(f"Jumlah harga tidak valid: {len(invalid_prices)}")

# Menampilkan baris datanya (jika ada)
if len(invalid_prices) > 0:
  display(invalid_prices)
else:
  print("there is no any line that identified as invalid_prices!")

Jumlah harga tidak valid: 0
there is no any line that identified as invalid_prices!


In [11]:
# latihan 3
# Menghitung jumlah transaksi per kategori
kategori_counts = df['category'].value_counts()

print("Jumlah transaksi per kategori:")
print(kategori_counts)

Jumlah transaksi per kategori:
category
Rumah Tangga    91
Kesehatan       86
Buku            81
Fashion         80
Elektronik      78
Olahraga        74
Name: count, dtype: Int64
